# UNet GAN Denoising — Main Notebook
학습(Train) → 단일 검증(Verify) → 3모델 비교(Compare) 순서로 실행합니다.

## 1. 환경 설정
Google Drive 마운트 후 GitHub에서 최신 소스 코드를 가져옵니다.

In [ ]:
import os
import sys
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

REPO_URL = 'https://github.com/sjjeon0925/UNet_GAN_Denoising'
REPO_DIR = '/content/UNet_GAN_Denoising'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, REPO_DIR)

from src.models import UNetMasker, UpgradedUNet
from src.audio_utils import get_spec, spec_to_wav
from src.metrics import calculate_metrics, get_random_test_samples, load_test_pair
from src.trainer import load_checkpoint, train_unet, train_unet_gan, train_upgraded_gan

print('환경 설정 완료.')

## 2. 경로 및 학습 설정
`MODEL_VERSION`을 변경하면 학습/검증에 사용할 모델이 바뀝니다.

In [ ]:
# 모델 버전 선택: 'unet' | 'unet_gan' | 'upgraded_gan'
MODEL_VERSION = 'upgraded_gan'

DATA_DIR  = '/content/drive/MyDrive/UNet'
MODEL_DIR = '/content/drive/MyDrive/UNet_GAN_Upgraded'

VOICE_ZIPS = [
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_01.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_02.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_03.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_04.zip',
    f'{DATA_DIR}/Korean_Voice/KsponSpeech_05.zip',
]
NOISE_ZIP      = f'{DATA_DIR}/Clean_Noise_Zips/Cleaned_Noise_All.zip'
VOICE_ZIP_TEST = VOICE_ZIPS[0]
CHECKPOINT_DIR = f'{MODEL_DIR}/Checkpoints'

config = {
    'voice_zips':     VOICE_ZIPS,
    'noise_zip':      NOISE_ZIP,
    'checkpoint_dir': CHECKPOINT_DIR,
    'noise_ext':      '.wav',
    'batch_size':     32,
    'epochs_per_zip': 10,
    'lr_g':           1e-4,
    'lr_d':           1e-5,
    'lambda_adv':     0.01,
    'num_workers':    2,
}

print(f'모델 버전: {MODEL_VERSION}')
print(f'체크포인트 경로: {CHECKPOINT_DIR}')

## 3. 학습 실행
이미 저장된 체크포인트가 있으면 이어서 학습합니다.

In [ ]:
if MODEL_VERSION == 'unet':
    train_unet(config)
elif MODEL_VERSION == 'unet_gan':
    train_unet_gan(config)
elif MODEL_VERSION == 'upgraded_gan':
    train_upgraded_gan(config)
else:
    raise ValueError(f'알 수 없는 MODEL_VERSION: {MODEL_VERSION}')

## 4. 단일 모델 검증 (10샘플 시각화)
학습된 모델로 랜덤 샘플 10개에 대한 스펙트로그램과 오디오를 저장합니다.

In [ ]:
import torch
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

RESULT_DIR = f'{MODEL_DIR}/Test_Results'
os.makedirs(RESULT_DIR, exist_ok=True)

ckpt_name_map = {
    'unet':         'unet_only_model.pth',
    'unet_gan':     'unet_gan_model.pth',
    'upgraded_gan': 'upgraded_gan_model.pth',
}
model_class_map = {
    'unet':         UNetMasker,
    'unet_gan':     UNetMasker,
    'upgraded_gan': UpgradedUNet,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = os.path.join(CHECKPOINT_DIR, ckpt_name_map[MODEL_VERSION])

model = model_class_map[MODEL_VERSION]().to(device)
load_checkpoint(ckpt_path, device, model)
model.eval()

for i in range(10):
    print(f'--- 샘플 {i + 1}/10 ---')
    v_path, n_path = get_random_test_samples(VOICE_ZIP_TEST, NOISE_ZIP)
    clean, noisy = load_test_pair(v_path, n_path)

    with torch.no_grad():
        noisy_t = torch.FloatTensor(noisy).unsqueeze(0).unsqueeze(0)
        clean_t = torch.FloatTensor(clean).unsqueeze(0).unsqueeze(0)
        mag, phase   = get_spec(noisy_t)
        clean_mag, _ = get_spec(clean_t)
        denoised_mag, mask = model(mag.unsqueeze(1).to(device))
        denoised_wav = spec_to_wav(denoised_mag.squeeze(1).cpu(), phase)[0].numpy()

    titles = ['Noisy Input', 'Predicted Mask', 'Denoised Output', 'Clean Target']
    specs  = [
        torch.log1p(mag.squeeze().cpu()),
        mask.squeeze().cpu(),
        torch.log1p(denoised_mag.squeeze().cpu()),
        torch.log1p(clean_mag.squeeze().cpu()),
    ]
    cmaps = ['magma', 'bone', 'magma', 'magma']

    fig, axes = plt.subplots(4, 1, figsize=(12, 13))
    for ax, title, spec, cmap in zip(axes, titles, specs, cmaps):
        ax.set_title(title)
        im = ax.imshow(spec.numpy(), aspect='auto', origin='lower', cmap=cmap)
        fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, f'spec_{i + 1}.png'))
    plt.show()

    sf.write(os.path.join(RESULT_DIR, f'test_{i + 1}_noisy.wav'),    noisy,        16000)
    sf.write(os.path.join(RESULT_DIR, f'test_{i + 1}_clean.wav'),    clean,        16000)
    sf.write(os.path.join(RESULT_DIR, f'test_{i + 1}_denoised.wav'), denoised_wav, 16000)

print(f'검증 완료. 결과 저장 위치: {RESULT_DIR}')

## 5. 3가지 모델 성능 비교
3개 모델의 체크포인트가 모두 `COMPARE_DIR`에 있어야 합니다.

In [ ]:
!pip install -q pesq pystoi

import torch
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from src.models import UNetMasker, UpgradedUNet

COMPARE_DIR        = f'{MODEL_DIR}/Comparation'
COMPARE_RESULT_DIR = f'{COMPARE_DIR}/Comparison_Results'
os.makedirs(COMPARE_RESULT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_info = {
    'UNet_Only':    {'path': f'{COMPARE_DIR}/unet_only_model.pth',    'class': UNetMasker},
    'UNet_GAN':     {'path': f'{COMPARE_DIR}/unet_gan_model.pth',     'class': UNetMasker},
    'Upgraded_GAN': {'path': f'{COMPARE_DIR}/upgraded_gan_model.pth', 'class': UpgradedUNet},
}

models = {}
for name, info in model_info.items():
    if os.path.exists(info['path']):
        m = info['class']().to(device)
        load_checkpoint(info['path'], device, m)
        m.eval()
        models[name] = m
        print(f'로드 완료: {name}')
    else:
        print(f'파일 없음 (건너뜀): {info["path"]}')

metrics_history = {name: {'PESQ': [], 'STOI': [], 'SNR': []} for name in models}

for i in range(10):
    print(f'--- 샘플 {i + 1}/10 ---')
    v_path, n_path = get_random_test_samples(VOICE_ZIP_TEST, NOISE_ZIP)
    clean, noisy   = load_test_pair(v_path, n_path)

    with torch.no_grad():
        noisy_t      = torch.FloatTensor(noisy).unsqueeze(0).unsqueeze(0)
        clean_t      = torch.FloatTensor(clean).unsqueeze(0).unsqueeze(0)
        mag, phase   = get_spec(noisy_t)
        clean_mag, _ = get_spec(clean_t)
        mag_dev      = mag.unsqueeze(1).to(device)

        results = {}
        for name, m in models.items():
            dm, _ = m(mag_dev)
            wav   = spec_to_wav(dm.squeeze(1).cpu(), phase)[0].numpy()
            results[name] = {'mag': dm.squeeze().cpu(), 'wav': wav}

    for name, res in results.items():
        p, s, snr = calculate_metrics(clean, res['wav'])
        metrics_history[name]['PESQ'].append(p)
        metrics_history[name]['STOI'].append(s)
        metrics_history[name]['SNR'].append(snr)
        print(f'  [{name}] PESQ: {p:.2f} | STOI: {s:.2f} | SNR: {snr:.2f}dB')

    n_rows = 2 + len(results)
    fig, axes = plt.subplots(n_rows, 1, figsize=(12, 4 * n_rows))
    axes[0].set_title('Noisy Input')
    axes[0].imshow(torch.log1p(mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
    for idx, (name, res) in enumerate(results.items(), start=1):
        axes[idx].set_title(f'Denoised ({name})')
        axes[idx].imshow(torch.log1p(res['mag']).numpy(), aspect='auto', origin='lower', cmap='magma')
    axes[-1].set_title('Clean Target')
    axes[-1].imshow(torch.log1p(clean_mag.squeeze().cpu()).numpy(), aspect='auto', origin='lower', cmap='magma')
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_RESULT_DIR, f'comparison_spec_{i + 1}.png'))
    plt.show()

    sf.write(os.path.join(COMPARE_RESULT_DIR, f'test_{i + 1}_noisy.wav'), noisy, 16000)
    sf.write(os.path.join(COMPARE_RESULT_DIR, f'test_{i + 1}_clean.wav'), clean, 16000)
    for name, res in results.items():
        sf.write(os.path.join(COMPARE_RESULT_DIR, f'test_{i + 1}_denoised_{name}.wav'), res['wav'], 16000)

avg = {name: {m: np.mean(v) for m, v in h.items()} for name, h in metrics_history.items()}
print('\n=== 최종 평균 성능 ===')
for name, m in avg.items():
    print(f'[{name}] PESQ: {m["PESQ"]:.3f} | STOI: {m["STOI"]:.3f} | SNR: {m["SNR"]:.3f}dB')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4C72B0', '#55A868', '#C44E52']
for idx, metric in enumerate(['PESQ', 'STOI', 'SNR']):
    values = [avg[name][metric] for name in models]
    axes[idx].bar(models.keys(), values, color=colors)
    axes[idx].set_title(f'Average {metric}')
    for j, v in enumerate(values):
        axes[idx].text(j, v * 1.02, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(COMPARE_RESULT_DIR, 'final_metrics_comparison.png'))
plt.show()
print(f'비교 완료. 결과 저장 위치: {COMPARE_RESULT_DIR}')